# Páramos

> **Contexto de dominio:** [`docs/fuentes/paramos.md`](../../docs/fuentes/paramos.md)  
> **Bloque:** A | **Línea:** `paramos`  
> **Variable principal:** `temperatura` (°C)  
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/paramos.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "paramos"
VARIABLE = "temperatura"
UNIDAD = "°C"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `paramos` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "paramos.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/paramos.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/paramos.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "temperatura": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: paramos -->

## 1b. Indicadores de paramo como fabrica de agua — IRH, caudal y cobertura

La temperatura es la variable de monitoreo climatico, pero los **indicadores de salud**
del paramo son hidrologicos y de cobertura vegetal:

| Indicador | Escala | Umbral critico | Fuente |
|---|---|---|---|
| **IRH** (Retencion Hidrica) | 0 a 1 | < 0.6 = deterioro | IDEAM / ENA |
| **Cobertura natural** | 0-100% | < 70% = alerta | Landsat/Sentinel |
| **Caudal base** | m3/s | Tendencia decreciente = alarma | IDEAM limnimetria |

> **Ley 1930/2018:** Los complejos de paramo delimitados deben mantener IRH >= 0.6
> para garantizar el suministro de agua a las cuencas abastecedoras.

> **IRH_THRESHOLDS en config.py:** muy_baja=0.2, baja=0.4, moderada=0.6, alta=0.8

In [ ]:
# Indicadores sinteticos de ecosistema paramo (complementan la temperatura)
np.random.seed(77)
n = len(df)

# IRH: Indice de Retencion Hidrica (0-1)
# Tendencia decreciente por presion agricola, recuperacion parcial en anos lluviosos
irh_trend = np.linspace(0.75, 0.58, n)  # deterioro gradual
irh_seasonal = 0.08 * np.sin(2 * np.pi * np.arange(n) / 12)  # ciclo lluvia-sequia
irh = np.clip(irh_trend + irh_seasonal + np.random.normal(0, 0.03, n), 0.1, 1.0)

# Caudal base (m3/s) — correlacionado con IRH y precipitacion
caudal = 2.5 * irh + 0.5 * np.random.normal(0, 0.2, n)
caudal = np.clip(caudal, 0.1, None)

# Cobertura natural (%) — decrece por ganaderia y papa
cobertura = np.clip(np.linspace(82, 68, n) + np.random.normal(0, 2, n), 40, 100)

df['irh'] = irh.round(3)
df['caudal_m3s'] = caudal.round(3)
df['cobertura_pct'] = cobertura.round(1)

from estadistica_ambiental.config import IRH_THRESHOLDS

print('IRH_THRESHOLDS (config.py):', IRH_THRESHOLDS)
print(f'\nIRH actual | min={irh.min():.2f} max={irh.max():.2f} media={irh.mean():.2f}')
print(f'Meses con IRH < 0.6 (deterioro): {(irh < 0.6).sum()} de {n}')
df[['fecha', 'temperatura', 'irh', 'caudal_m3s', 'cobertura_pct']].head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `paramos` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_paramos.html",
       title="EDA — Páramos", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "temperatura", title="Páramos — temperatura (°C)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "temperatura", period="month")
plt.show()

## 3c. Dashboard IRH — Salud hidrologica del paramo

El **IRH** (Indice de Retencion y Regulacion Hidrica) es el indicador operativo
para clasificar el estado del paramo como regulador hidrico. Las categorias provienen
de `config.IRH_THRESHOLDS` y son las mismas que usa el IDEAM en el ENA (Estudio Nacional del Agua):

| IRH | Categoria | Significado para gestion |
|---|---|---|
| > 0.8 | Muy alta | Ecosistema pristino — preservar |
| 0.6 – 0.8 | Alta / moderada | Funcionamiento aceptable |
| 0.4 – 0.6 | Baja | Deterioro — iniciar restauracion |
| < 0.4 | Muy baja | Crisis hidrica — intervencion urgente |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel A: IRH en el tiempo con bandas de umbral
ax = axes[0, 0]
ax.plot(df['fecha'], df['irh'], lw=1.5, color='#2980b9', label='IRH mensual')
ax.axhspan(0.8, 1.0, alpha=0.12, color='#27ae60', label='Muy alta (>0.8)')
ax.axhspan(0.6, 0.8, alpha=0.10, color='#f1c40f', label='Alta/Moderada (0.6-0.8)')
ax.axhspan(0.4, 0.6, alpha=0.10, color='#e67e22', label='Baja (0.4-0.6)')
ax.axhspan(0.0, 0.4, alpha=0.12, color='#e74c3c', label='Muy baja (<0.4)')
ax.axhline(0.6, color='#e67e22', ls='--', lw=1.5, label='Umbral critico 0.6')
ax.set_title('IRH — Retencion Hidrica del Paramo', fontweight='bold')
ax.set_ylabel('IRH (0-1)'); ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc='lower left'); ax.grid(alpha=0.3)

# Panel B: Cobertura natural (%)
ax = axes[0, 1]
ax.fill_between(df['fecha'], df['cobertura_pct'], alpha=0.6, color='#27ae60')
ax.axhline(70, color='red', ls='--', lw=1.5, label='Umbral critico 70%')
ax.set_title('Cobertura Natural (%)', fontweight='bold')
ax.set_ylabel('%'); ax.set_ylim(50, 100)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel C: Caudal base (m3/s)
ax = axes[1, 0]
ax.plot(df['fecha'], df['caudal_m3s'], lw=1, color='#3498db')
ax.fill_between(df['fecha'], df['caudal_m3s'], alpha=0.3, color='#3498db')
ax.set_title('Caudal Base (m3/s)', fontweight='bold')
ax.set_ylabel('m3/s'); ax.grid(alpha=0.3)

# Panel D: IRH vs Cobertura (scatter)
ax = axes[1, 1]
sc = ax.scatter(df['cobertura_pct'], df['irh'], c=df['temperatura'],
                cmap='RdYlGn_r', alpha=0.6, s=20)
plt.colorbar(sc, ax=ax, label='Temp (C)')
ax.axhline(0.6, color='orange', ls='--', lw=1)
ax.axvline(70, color='red', ls='--', lw=1)
ax.set_xlabel('Cobertura Natural (%)'); ax.set_ylabel('IRH')
ax.set_title('IRH vs Cobertura (color=temperatura)', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Dashboard de Salud Hidrologica — Paramo (Ley 1930/2018)',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

# Resumen de alertas
n_alerta = (df['irh'] < 0.6).sum()
n_cob = (df['cobertura_pct'] < 70).sum()
print(f'Meses con IRH < 0.6 (restauracion necesaria): {n_alerta}/{len(df)} ({n_alerta/len(df)*100:.0f}%)')
print(f'Meses con cobertura < 70% (alerta): {n_cob}/{len(df)} ({n_cob/len(df)*100:.0f}%)')

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `paramos` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para Páramos) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["temperatura"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5c. Gradiente altitudinal Caldas-Lang — temperatura vs. altitud

La **relacion Caldas-Lang** describe la variacion de temperatura con la altitud en Colombia.
El gradiente termico vertical es ~0.6C/100m en paramos andinos (lapse rate):

```
T(altitud) = T_referencia - lapse_rate * (altitud - altitud_ref) / 100
lapse_rate tipico en paramos: 0.55 - 0.65 C/100m
Rango altitudinal paramo: 3.000 - 4.700 m.s.n.m.
```

Esta relacion es util para:
- Extrapolar temperatura a zonas sin estaciones de alta montana
- Delimitar la franja de transicion bosque andino - paramo (~3.000 m)
- Proyectar la franja de paramo bajo escenarios de calentamiento

In [ ]:
# Gradiente Caldas-Lang: temperatura ~ altitud en paramo andino colombiano
ALTITUDES = np.arange(2500, 4800, 100)  # m.s.n.m.
LAPSE_RATE = 0.60  # C/100m — gradiente tipico paramo andino
ALT_REF = 3000     # m.s.n.m. referencia
T_REF = 12.5       # C en 3000 m (limite inferior paramo)

T_teorico = T_REF - LAPSE_RATE * (ALTITUDES - ALT_REF) / 100

# Simular estaciones reales con variabilidad
np.random.seed(55)
n_estaciones = 20
alt_est = np.random.choice(ALTITUDES, n_estaciones)
T_est   = T_REF - LAPSE_RATE * (alt_est - ALT_REF)/100 + np.random.normal(0, 0.8, n_estaciones)

# Regresion lineal altitud-temperatura
from numpy.polynomial import polynomial as P
coef = np.polyfit(alt_est, T_est, 1)
lapse_estimado = -coef[0] * 100  # C/100m

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T_teorico, ALTITUDES, lw=2, color='#2980b9', label=f'Gradiente teorico ({LAPSE_RATE} C/100m)')
ax.scatter(T_est, alt_est, color='#e74c3c', s=60, zorder=5, label='Estaciones IDEAM (simuladas)')
T_fit = np.polyval(coef, ALTITUDES)
ax.plot(T_fit, ALTITUDES, ls='--', color='#e74c3c', lw=1.5,
        label=f'Ajuste empirico ({lapse_estimado:.2f} C/100m)')

# Bandas altitudinales
ax.axhspan(3000, 3500, alpha=0.08, color='#27ae60', label='Subparamo (3.000-3.500 m)')
ax.axhspan(3500, 4200, alpha=0.08, color='#f1c40f', label='Paramo abierto (3.500-4.200 m)')
ax.axhspan(4200, 4800, alpha=0.08, color='#3498db', label='Superparamo (>4.200 m)')

ax.set_xlabel('Temperatura media (C)'); ax.set_ylabel('Altitud (m.s.n.m.)')
ax.set_title('Relacion Caldas-Lang — Gradiente termico altitudinal en paramo andino',
             fontweight='bold')
ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Gradiente empirico estimado: {lapse_estimado:.3f} C/100m')
T_3500 = T_REF - LAPSE_RATE * (3500 - ALT_REF)/100
T_4200 = T_REF - LAPSE_RATE * (4200 - ALT_REF)/100
print(f'T estimada a 3.500 m: {T_3500:.1f} C | a 4.200 m: {T_4200:.1f} C')
print(f'Con calentamiento +1.5C (escenario SSP2-4.5): franja paramo sube ~250 m')

## 5b. Análisis de excedencias normativas
> Compara `temperatura` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["temperatura"], variable="temperatura")
if rep.empty:
    print("Sin normas colombianas registradas para 'temperatura'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["temperatura"], method="linear")
print(f"Faltantes antes: {df["temperatura"].isna().sum()} | "
      f"después: {df_clean["temperatura"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["temperatura"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: paramos -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Páramos / alta montaña

- **Gradiente Caldas-Lang:** clasificación bioclimática por altitud (T°, P) — usar para validar coherencia de muestras.
- **Lag ENSO = 2 meses** (respuesta rápida a precipitación en alta montaña).
- **Variabilidad espacial alta:** kriging con variograma exponencial; verificar Moran's I antes.
- **Cobertura:** delimitación 1:25.000 (Atlas Páramos IAvH).

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Páramos (`paramos`)
- **Variable analizada:** `temperatura` (°C)
- **Modelos ejecutados:** SARIMA, PROPHET, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/paramos.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.